# Exploration et analyse du catalogue Netflix.

> Audit de la qualité des données

In [345]:
import numpy as np
import pandas as pd

In [346]:
netflix = pd.read_csv("../data/raw/netflix_titles_raw.csv")

## Analyse de la structure

### Informations sur le dataset

In [347]:
netflix.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB


### Description statistiques du dataset

In [348]:
netflix.describe()

,release_year
count,8807.000000
mean,2014.180198
std,8.819312
min,1925.000000
25%,2013.000000
50%,2017.000000
75%,2019.000000
max,2021.000000


### Autres informations sur le dataset (La taille, le nombre de cellules et les dtypes)

In [349]:
netflix.shape

(8807, 12)

In [350]:
netflix.size

105684

In [351]:
netflix.dtypes

show_id           str
type              str
title             str
director          str
cast              str
country           str
date_added        str
release_year    int64
rating            str
duration          str
listed_in         str
description       str
dtype: object

## Analyse des valeurs manquantes

- Détecter les valeurs manquantes
- Compter le nombre de valeurs manquantes
- Calculer le pourcentage de valeurs manquantes

In [352]:
netflix.isna().sum()

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

In [353]:
netflix.isna().mean()

show_id         0.000000
type            0.000000
title           0.000000
director        0.299080
cast            0.093675
country         0.094357
date_added      0.001135
release_year    0.000000
rating          0.000454
duration        0.000341
listed_in       0.000000
description     0.000000
dtype: float64

#### Hypothèse

Est-ce que `director` est vide parce que la donnée n'a jamais été collectée (MCAR — manquant complètement au hasard), ou parce qu'il existe une vraie raison structurelle (par exemple : les TV Shows n'ont souvent pas de "réalisateur" unique désigné, contrairement aux films) ?

On distingue généralement trois types de données manquantes :

- MCAR (Missing Completely At Random) : la probabilité d'absence n'a aucun lien avec les données elles-mêmes ou d'autres variables (Rare en pratique).

- MAR (Missing At Random) : l'absence est liée à une autre colonne observable (ex : director manquant plus souvent quand type == "TV Show").

- MNAR (Missing Not At Random) : l'absence est liée à la valeur elle-même, non observable (ex : les gens à très haut revenu qui refusent de déclarer leur salaire).

> Pourquoi ça compte concrètement : si tu supprimes ou imputes sans comprendre le mécanisme, tu peux introduire un biais silencieux dans tes données. Si `director` manque majoritairement pour les séries TV, et que tu drop toutes les lignes avec director manquant, tu viens de supprimer presque toutes tes séries TV du dataset — sans le vouloir.

In [354]:
netflix[netflix["type"] == "TV Show"].isna().sum()

show_id            0
type               0
title              0
director        2446
cast             350
country          391
date_added        10
release_year       0
rating             2
duration           0
listed_in          0
description        0
dtype: int64

In [355]:
netflix[netflix["type"] == "TV Show"].isna().mean()

show_id         0.000000
type            0.000000
title           0.000000
director        0.914051
cast            0.130792
country         0.146114
date_added      0.003737
release_year    0.000000
rating          0.000747
duration        0.000000
listed_in       0.000000
description     0.000000
dtype: float64

##### Pour compléter la vérification de l'hypothèse

Taux conditionnel P(manquant | TV Show) vs Proportion P(TV Show | manquant)).

Avant la conclusion définitivement, faire le même calcul côté Movie sur director uniquement. Si le taux est très bas pour Movie et très haut pour TV Show, alors l'hypothèse MAR est solidement confirmée — c'est le contraste entre les deux groupes qui fait la preuve, pas juste un chiffre isolé.

In [356]:
netflix[netflix["type"] == "Movie"].isna().sum()

show_id           0
type              0
title             0
director        188
cast            475
country         440
date_added        0
release_year      0
rating            2
duration          3
listed_in         0
description       0
dtype: int64

In [357]:
netflix[netflix["type"] == "Movie"].isna().mean()

show_id         0.000000
type            0.000000
title           0.000000
director        0.030664
cast            0.077475
country         0.071766
date_added      0.000000
release_year    0.000000
rating          0.000326
duration        0.000489
listed_in       0.000000
description     0.000000
dtype: float64

Sur le contraste director — 3,07% pour Movie vs 91,4% pour TV Show. Ça, c'est une preuve solide et propre d'un mécanisme MAR : l'absence de director est quasi exclusivement structurelle (les séries n'ont généralement pas de réalisateur unique crédité), pas aléatoire. Tu peux documenter cette conclusion telle quelle dans ton rapport final, avec les deux chiffres côte à côte — c'est exactement le type de justification qu'on attend.

#### Hypothèse

Est ce que `cast` est vide parce que les documentaires et docuseries n'ont pas de casting ?


In [358]:
netflix[netflix["listed_in"].str.contains("Documentaries")].isna().mean()

show_id         0.000000
type            0.000000
title           0.000000
director        0.065593
cast            0.487917
country         0.086306
date_added      0.000000
release_year    0.000000
rating          0.000000
duration        0.000000
listed_in       0.000000
description     0.000000
dtype: float64

In [359]:
netflix[netflix["listed_in"].str.contains("Documentaries")].isna().sum()

show_id           0
type              0
title             0
director         57
cast            424
country          75
date_added        0
release_year      0
rating            0
duration          0
listed_in         0
description       0
dtype: int64

In [360]:
netflix[netflix["listed_in"].str.contains("Docuseries")].isna().mean()

show_id         0.000000
type            0.000000
title           0.000000
director        0.848101
cast            0.524051
country         0.164557
date_added      0.002532
release_year    0.000000
rating          0.000000
duration        0.000000
listed_in       0.000000
description     0.000000
dtype: float64

In [361]:
netflix[netflix["listed_in"].str.contains("Docuseries")].isna().sum()

show_id           0
type              0
title             0
director        335
cast            207
country          65
date_added        1
release_year      0
rating            0
duration          0
listed_in         0
description       0
dtype: int64

Si ce nombre est 0, ton addition simple était correcte. S'il est différent de 0, il faut le soustraire de ta somme pour ne pas compter ces lignes deux fois (principe d'inclusion-exclusion : `|A ∪ B| = |A| + |B| - |A ∩ B|`).

In [362]:
424 + 207 + 0

631

##### Pour compléter la vérification de l'hypothèse

Hypothèse MAR confirmée à 76%, le reste (24%) reste sans explication claire pour l'instant, ce qui est normal

In [363]:
netflix[
    netflix["listed_in"].str.contains("Documentaries")
    & netflix["listed_in"].str.contains("Docuseries")
].shape[0]

0

In [364]:
631 / 825

0.7648484848484849

In [365]:
netflix[netflix["duration"].isna()]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
5541,s5542,Movie,Louis C.K. 2017,Louis C.K.,Louis C.K.,United States,"April 4, 2017",2017,74 min,NaN,Movies,"Louis C.K. muses on religion, eternal love, gi..."
5794,s5795,Movie,Louis C.K.: Hilarious,Louis C.K.,Louis C.K.,United States,"September 16, 2016",2010,84 min,NaN,Movies,Emmy-winning comedy writer Louis C.K. brings h...
5813,s5814,Movie,Louis C.K.: Live at the Comedy Store,Louis C.K.,Louis C.K.,United States,"August 15, 2016",2015,66 min,NaN,Movies,The comic puts his trademark hilarious/thought...


Tableau `duration` manquant (3 lignes) : les colonnes `rating` contiennent `"74 min", "84 min", "66 min"` — ce sont clairement des durées, pas des ratings (`P`G-13, TV-MA...). Et les 3 lignes sont toutes des specials de `Louis C.K.` Ce n'est pas un hasard : on peut parier que dans le CSV source, le champ rating était vide pour ces lignes précises (peut-être parce que ces contenus n'ont jamais reçu de classification officielle), et un bug de génération du dataset a décalé la valeur de duration vers la gauche pour combler le vide — laissant duration vide à la fin. Une vraie corruption structurelle, localisée à 3 lignes.

In [366]:
mask = netflix["duration"].isna()
netflix.loc[mask, "duration"] = netflix.loc[mask, "rating"]
netflix.loc[mask, "rating"] = np.nan
netflix.loc[mask]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
5541,s5542,Movie,Louis C.K. 2017,Louis C.K.,Louis C.K.,United States,"April 4, 2017",2017,NaN,74 min,Movies,"Louis C.K. muses on religion, eternal love, gi..."
5794,s5795,Movie,Louis C.K.: Hilarious,Louis C.K.,Louis C.K.,United States,"September 16, 2016",2010,NaN,84 min,Movies,Emmy-winning comedy writer Louis C.K. brings h...
5813,s5814,Movie,Louis C.K.: Live at the Comedy Store,Louis C.K.,Louis C.K.,United States,"August 15, 2016",2015,NaN,66 min,Movies,The comic puts his trademark hilarious/thought...


In [367]:
netflix[netflix["rating"].isna()]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
5541,s5542,Movie,Louis C.K. 2017,Louis C.K.,Louis C.K.,United States,"April 4, 2017",2017,NaN,74 min,Movies,"Louis C.K. muses on religion, eternal love, gi..."
5794,s5795,Movie,Louis C.K.: Hilarious,Louis C.K.,Louis C.K.,United States,"September 16, 2016",2010,NaN,84 min,Movies,Emmy-winning comedy writer Louis C.K. brings h...
5813,s5814,Movie,Louis C.K.: Live at the Comedy Store,Louis C.K.,Louis C.K.,United States,"August 15, 2016",2015,NaN,66 min,Movies,The comic puts his trademark hilarious/thought...
5989,s5990,Movie,13TH: A Conversation with Oprah Winfrey & Ava ...,NaN,"Oprah Winfrey, Ava DuVernay",NaN,"January 26, 2017",2017,NaN,37 min,Movies,Oprah Winfrey sits down with director Ava DuVe...
6827,s6828,TV Show,Gargantia on the Verdurous Planet,NaN,"Kaito Ishikawa, Hisako Kanemoto, Ai Kayano, Ka...",Japan,"December 1, 2016",2013,NaN,1 Season,"Anime Series, International TV Shows","After falling through a wormhole, a space-dwel..."
7312,s7313,TV Show,Little Lunch,NaN,"Flynn Curry, Olivia Deeble, Madison Lu, Oisín ...",Australia,"February 1, 2018",2015,NaN,1 Season,"Kids' TV, TV Comedies","Adopting a child's perspective, this show take..."
7537,s7538,Movie,My Honor Was Loyalty,Alessandro Pepe,"Leone Frisa, Paolo Vaccarino, Francesco Miglio...",Italy,"March 1, 2017",2015,NaN,115 min,Dramas,"Amid the chaos and horror of World War II, a c..."


Tableau `rating` manquant (4 lignes) : regarde bien — ici duration contient des valeurs parfaitement normales (`37 min, 1 Season, 115 min...`). Il n'y a aucun décalage. C'est juste rating qui est réellement absent, sans lien avec le bug repéré dans l'autre tableau. Deux causes différentes, deux corrections différentes.

#### Suppression vs Imputation (Remplacement des valeurs manquantes avec des valeurs explicites)

> Ne jamais supprimer une ligne entière à cause d'une seule colonne problématique quand le reste de la ligne a de la valeur. La suppression (dropna) ne devrait être qu'un dernier recours, pas un réflexe.

Petite nuance à ajouter pour les justifications futures (dans le rapport de nettoyage) : le choix "Unknown" plutôt que dropna() est d'autant plus justifié ici que le volume est négligeable (7/8807 ≈ 0,08%) — même si l'hypothèse d'imputation était légèrement imparfaite, l'impact sur une analyse globale serait quasi nul. C'est un raisonnement de risque : plus le volume concerné est petit, moins il faut être rigoureux sur le choix d'imputation (l'erreur potentielle est diluée) ; plus il est grand (comme director à 30%), plus la stratégie doit être réfléchie car une erreur là aurait un vrai impact statistique.

In [368]:
netflix[netflix["date_added"].isna()]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
6066,s6067,TV Show,A Young Doctor's Notebook and Other Stories,NaN,"Daniel Radcliffe, Jon Hamm, Adam Godley, Chris...",United Kingdom,NaN,2013,TV-MA,2 Seasons,"British TV Shows, TV Comedies, TV Dramas","Set during the Russian Revolution, this comic ..."
6174,s6175,TV Show,Anthony Bourdain: Parts Unknown,NaN,Anthony Bourdain,United States,NaN,2018,TV-PG,5 Seasons,Docuseries,This CNN original series has chef Anthony Bour...
6795,s6796,TV Show,Frasier,NaN,"Kelsey Grammer, Jane Leeves, David Hyde Pierce...",United States,NaN,2003,TV-PG,11 Seasons,"Classic & Cult TV, TV Comedies",Frasier Crane is a snooty but lovable Seattle ...
6806,s6807,TV Show,Friends,NaN,"Jennifer Aniston, Courteney Cox, Lisa Kudrow, ...",United States,NaN,2003,TV-14,10 Seasons,"Classic & Cult TV, TV Comedies",This hit sitcom follows the merry misadventure...
6901,s6902,TV Show,Gunslinger Girl,NaN,"Yuuka Nanri, Kanako Mitsuhashi, Eri Sendai, Am...",Japan,NaN,2008,TV-14,2 Seasons,"Anime Series, Crime TV Shows","On the surface, the Social Welfare Agency appe..."
7196,s7197,TV Show,Kikoriki,NaN,Igor Dmitriev,NaN,NaN,2010,TV-Y,2 Seasons,Kids' TV,A wacky rabbit and his gang of animal pals hav...
7254,s7255,TV Show,La Familia P. Luche,NaN,"Eugenio Derbez, Consuelo Duval, Luis Manuel Áv...",United States,NaN,2012,TV-14,3 Seasons,"International TV Shows, Spanish-Language TV Sh...","This irreverent sitcom featues Ludovico, Feder..."
7406,s7407,TV Show,Maron,NaN,"Marc Maron, Judd Hirsch, Josh Brener, Nora Zeh...",United States,NaN,2016,TV-MA,4 Seasons,TV Comedies,"Marc Maron stars as Marc Maron, who interviews..."
7847,s7848,TV Show,Red vs. Blue,NaN,"Burnie Burns, Jason Saldaña, Gustavo Sorola, G...",United States,NaN,2015,NR,13 Seasons,"TV Action & Adventure, TV Comedies, TV Sci-Fi ...","This parody of first-person shooter games, mil..."
8182,s8183,TV Show,The Adventures of Figaro Pho,NaN,"Luke Jurevicius, Craig Behenna, Charlotte Haml...",Australia,NaN,2015,TV-Y7,2 Seasons,"Kids' TV, TV Comedies","Imagine your worst fears, then multiply them: ..."


## Analyse des types mixtes

In [369]:
netflix["duration"].unique()

<StringArray>
[   '90 min', '2 Seasons',  '1 Season',    '91 min',   '125 min', '9 Seasons',
   '104 min',   '127 min', '4 Seasons',    '67 min',
 ...
    '43 min',   '200 min',   '196 min',   '167 min',   '178 min',   '228 min',
    '18 min',   '205 min',   '201 min',   '191 min']
Length: 220, dtype: str

In [370]:
netflix[["type", "duration"]].sample(10)

,type,duration
7449,TV Show,1 Season
726,TV Show,1 Season
6098,Movie,89 min
1650,Movie,87 min
6493,Movie,91 min
4371,TV Show,1 Season
2960,Movie,92 min
5318,Movie,100 min
6129,Movie,92 min
3926,Movie,91 min


## Analyse des dates

In [371]:
netflix["date_added"].head(10).tolist()

['September 25, 2021',
 'September 24, 2021',
 'September 24, 2021',
 'September 24, 2021',
 'September 24, 2021',
 'September 24, 2021',
 'September 24, 2021',
 'September 24, 2021',
 'September 24, 2021',
 'September 24, 2021']

In [372]:
netflix["date_added"].dtype

<StringDtype(storage='python', na_value=nan)>

In [373]:
netflix["date_added"].isna().sum()

np.int64(10)

In [374]:
format_date = pd.to_datetime(netflix["date_added"], errors="coerce")
format_date

0      2021-09-25
1      2021-09-24
2      2021-09-24
3      2021-09-24
4      2021-09-24
          ...    
8802   2019-11-20
8803   2019-07-01
8804   2019-11-01
8805   2020-01-11
8806   2019-03-02
Name: date_added, Length: 8807, dtype: datetime64[us]

In [375]:
format_date.isna().sum()

np.int64(98)

- le nombre de dates invalides

In [376]:
98 - 10

88

In [377]:
netflix["date_added"].notna().sum()

np.int64(8797)

In [378]:
netflix[netflix["date_added"].notna() & format_date.isna()].shape[0]

88

In [379]:
netflix[netflix["date_added"].notna() & format_date.isna()]["date_added"].tolist()

[' August 4, 2017',
 ' December 23, 2018',
 ' December 15, 2018',
 ' July 1, 2017',
 ' July 26, 2019',
 ' May 26, 2016',
 ' November 1, 2019',
 ' December 2, 2017',
 ' March 15, 2019',
 ' October 1, 2019',
 ' December 15, 2017',
 ' July 1, 2017',
 ' August 4, 2017',
 ' April 4, 2017',
 ' December 28, 2016',
 ' March 31, 2018',
 ' February 1, 2019',
 ' January 1, 2018',
 ' July 1, 2017',
 ' February 24, 2018',
 ' March 31, 2018',
 ' December 14, 2018',
 ' July 20, 2018',
 ' January 17, 2018',
 ' September 7, 2016',
 ' October 31, 2018',
 ' August 21, 2017',
 ' October 8, 2013',
 ' November 1, 2017',
 ' December 1, 2018',
 ' March 16, 2016',
 ' December 15, 2015',
 ' September 8, 2015',
 ' May 28, 2016',
 ' December 3, 2019',
 ' November 1, 2016',
 ' January 1, 2018',
 ' November 3, 2015',
 ' December 1, 2019',
 ' February 1, 2019',
 ' October 1, 2019',
 ' May 1, 2018',
 ' April 17, 2016',
 ' March 22, 2015',
 ' November 1, 2019',
 ' September 16, 2018',
 ' April 16, 2019',
 ' June 1, 20

In [380]:
netflix["date_added"] = pd.to_datetime(
    netflix["date_added"].str.strip(), format="%B %d, %Y", errors="raise"
)

In [381]:
netflix["date_added"].isna().sum()

np.int64(10)

In [382]:
netflix[format_date.isna()].shape[0]

98

- Séparer les mois et les années dans 2 colonnes distinctes

In [383]:
netflix["month_added"] = netflix["date_added"].dt.month.astype(pd.Int64Dtype())
netflix["year_added"] = netflix["date_added"].dt.year.astype(pd.Int64Dtype())

netflix[["date_added", "month_added", "year_added"]]

,date_added,month_added,year_added
0,2021-09-25,9,2021
1,2021-09-24,9,2021
2,2021-09-24,9,2021
3,2021-09-24,9,2021
4,2021-09-24,9,2021
...,...,...,...
8802,2019-11-20,11,2019
8803,2019-07-01,7,2019
8804,2019-11-01,11,2019
8805,2020-01-11,1,2020


In [384]:
netflix[["date_added", "month_added", "year_added"]].dtypes

date_added     datetime64[us]
month_added             Int64
year_added              Int64
dtype: object

In [385]:
netflix[netflix["month_added"].isnull() & netflix["year_added"].isnull()][
    ["date_added", "month_added", "year_added"]
]

,date_added,month_added,year_added
6066,NaT,<NA>,<NA>
6174,NaT,<NA>,<NA>
6795,NaT,<NA>,<NA>
6806,NaT,<NA>,<NA>
6901,NaT,<NA>,<NA>
7196,NaT,<NA>,<NA>
7254,NaT,<NA>,<NA>
7406,NaT,<NA>,<NA>
7847,NaT,<NA>,<NA>
8182,NaT,<NA>,<NA>


In [386]:
netflix.sample(10)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,month_added,year_added
6877,s6878,Movie,Good Luck,Tarek Abdel Moaty,"Ahmed Eid, May Kassab, Diaa El Merghany, Samy ...",Egypt,2019-06-21,2012,TV-MA,99 min,"Dramas, International Movies","Seeking an apartment to share with his wife, a...",6,2019
3084,s3085,TV Show,Rainbow Ruby,NaN,"Alyssya Swales, Shannon Chan-Kent, Brian Drumm...","South Korea, Canada, United States, China",2019-12-31,2016,TV-Y,1 Season,Kids' TV,Ruby makes magical journeys with her teddy bea...,12,2019
429,s430,Movie,Sanitation Day,Seyi Babatope,"Blossom Chukwujekwu, Elozonam Ogbolu, Nse Ikpe...",Nigeria,2021-07-21,2020,TV-14,105 min,"Dramas, International Movies, Thrillers",Two cops must contend with the uncooperative t...,7,2021
5940,s5941,TV Show,Breaking Bad,NaN,"Bryan Cranston, Aaron Paul, Anna Gunn, Dean No...",United States,2013-08-02,2013,TV-MA,5 Seasons,"Crime TV Shows, TV Dramas, TV Thrillers",A high school chemistry teacher dying of cance...,8,2013
3623,s3624,Movie,Enter the Anime,Alex Burunova,"Kozo Morishita, Yoko Takahashi, Shinji Aramaki...","United States, Japan",2019-08-05,2019,TV-MA,59 min,"Documentaries, International Movies",What is anime? Through deep-dives with notable...,8,2019
5701,s5702,Movie,Interrogation,Vetrimaaran,"Dinesh Ravi, Samuthirakani, Kishore Kumar, Ana...",India,2016-12-01,2015,TV-MA,116 min,"Dramas, International Movies, Thrillers","In this fact-based drama, politically motivate...",12,2016
1685,s1686,Movie,Ainu Mosir,Takeshi Fukunaga,"Kanto Shimokura, Debo Akibe, Emi Shimokura, To...","United States, Japan",2020-11-17,2020,TV-PG,84 min,"Dramas, International Movies",A sensitive Ainu teen searches for a spiritual...,11,2020
4298,s4299,Movie,Once in a Lifetime Sessions with George Ezra,NaN,NaN,United States,2018-12-15,2018,TV-MA,78 min,"Documentaries, Music & Musicals",English singer-songwriter George Ezra performs...,12,2018
2680,s2681,Movie,Puerto Ricans in Paris,Ian Edelman,"Luis Guzmán, Edgar Garcia, Alice Taglioni, Ros...",United States,2020-04-16,2015,R,82 min,Comedies,New York police detectives Luis and Eddie expe...,4,2020
386,s387,TV Show,Mighty Express,NaN,"Jay Hatton, Zoe Hatz, Meesha Contreras, Dylan ...","United States, Canada",2021-07-27,2021,TV-Y,4 Seasons,Kids' TV,Catch a ride with the Mighty Express — a team ...,7,2021


## Analyse des valeurs dupliquées